# reshape-back — ex1: reshape_back — restore x's original shape

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reshape-back`. Running the final beacon cell reports progress against the `Backprop: reshape_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: reshape_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reshape-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reshape-back"
DD_SUBTOPIC = "Backprop: reshape_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `reshape_back` — quick refresher

`reshape(x, new_shape)` is a pure view rearrangement — every output entry corresponds 1-to-1 with an input entry, just at a different position. The local Jacobian is a permutation matrix, so the backward fn is **reshape `grad_out` back to `x`'s original shape**.

**Worked exemplar.**
```
x.shape         = (2, 6)
out = x.reshape(3, 4)
grad_out.shape  = (3, 4)
grad_in = grad_out.reshape(2, 6)  # restore x's shape
```

The forward shape is read from `x.shape`; no kwargs needed.

### Exercise 1 — reshape_back — restore x's original shape

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the view-op backward pattern: reshape grad_out back to x's original shape using x.shape read from the cached input.
> Keywords: reshape, shape-restore, view-op
> ```

**KCs targeted:** `reshape-backward-pattern`, `backward-fn-signature`

Implement `reshape_back(grad_out, out, x, new_shape)` for the forward op `out = x.reshape(new_shape)`.

Derivation:
- Reshape is a pure permutation of the storage — every output entry corresponds to exactly one input entry.
- The local Jacobian is a permutation matrix, so the backward fn is the inverse reshape: `grad_in = grad_out.reshape(x.shape)`.

The `new_shape` argument is part of the forward signature (so the wrapper passes it through) but you don't actually need it on the backward — `x.shape` is the authoritative shape to restore.

Return a `torch.Tensor` with the same shape as `x`. No autograd.

In [ ]:
def reshape_back(grad_out: Tensor, out: Tensor, x: Tensor, new_shape: tuple) -> Tensor:
    # Reshape is a permutation of storage; backward = inverse reshape.
    # x.shape is the authoritative target — new_shape is unused here.
    return grad_out.reshape(x.shape)


<details><summary>Solution</summary>

```python
def reshape_back(grad_out: Tensor, out: Tensor, x: Tensor, new_shape: tuple) -> Tensor:
    # Reshape is a permutation of storage; backward = inverse reshape.
    # x.shape is the authoritative target — new_shape is unused here.
    return grad_out.reshape(x.shape)
```

**Why `x.shape`, not `new_shape`.** They're inverses, but `x.shape` is direct: the target we want `grad_in` to wear. Using `new_shape` would force you to also store `x.shape` somewhere else.

**View ops all look like this.** Anything that's just a storage re-interpretation (`reshape`, `view`, `flatten`, `squeeze`, `unsqueeze`) has the same shape: backward = inverse-of-the-shape-op. The values don't change, only the layout.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()